# 01 Base revenue per car-day: exploration

**Question.** What is the base revenue $b_{s,k}$ per car-day, how noisy is it, and does it depend on anything
besides station $s$ and category $k$?

**Why the history gives base directly.** Appendix B: the history shows demand at the base price (no demand lost),
and with R8 $q=1 \Rightarrow$ price factor 1. So for a rental $i$ with revenue $R_i$ and length $L_i$ (car-days, both
ends counted), $R_i/L_i$ is a draw of base revenue per car-day. The natural estimator is car-day weighted:
$$\hat b_{s,k} = \frac{\sum_{i\in(s,k)} R_i}{\sum_{i\in(s,k)} L_i}.$$
R8 is linear in base, so if base is independent of demand, $E[\text{revenue}]$ only needs $E[b]$ (see [[Project Plan]]).
Section 6 checks that independence.

Data: `completed` rentals (8 weeks before day 1). `open` rentals are used only to check they price the same.
Future bookings (`booked`) are not used anywhere in this notebook.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Work from Project_4/notebooks or from Project_4 itself.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from fleetkit import load_instance  # noqa: E402
from fleetopt import level0  # noqa: E402

inst = load_instance(data_dir=ROOT / "data")
CATS = level0.categories_in_order(inst)                      # economy < compact < midsize < premium
CAT_COLOR = dict(zip(CATS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))
TYPE_COLOR = {"airport": "#2a78d6", "downtown": "#eb6834"}
ST = inst.stations.set_index("station_id")
STATIONS = ST.index.tolist()
ST_LABEL = {s: f"{s} {ST.loc[s, 'station_name']}\n({ST.loc[s, 'station_type']})" for s in STATIONS}
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
DAY1 = level0.horizon_dates(inst)[0]                         # 2027-03-01, a Monday

plt.rcParams.update({
    "figure.dpi": 90, "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": "#e4e3df", "grid.linewidth": 0.6,
    "axes.axisbelow": True, "axes.edgecolor": "#9a9993",
})
pd.set_option("display.width", 200, "display.max_columns", 40)

# One row per reservation, with the derived columns we use everywhere.
r = inst.reservations.copy()
for c in ["pickup_date", "return_date", "booking_date"]:
    r[c] = pd.to_datetime(r[c])
r["length"] = (r["return_date"] - r["pickup_date"]).dt.days + 1     # car-days, both ends counted
r["lead"] = (r["pickup_date"] - r["booking_date"]).dt.days           # days booked ahead
r["rev_per_day"] = r["revenue"] / r["length"]
r["weekday"] = pd.Categorical(r["pickup_date"].dt.day_name().str[:3], WEEKDAYS, ordered=True)
r["week"] = (r["pickup_date"] - r["pickup_date"].min()).dt.days // 7 + 1
r["station_type"] = r["pickup_station_id"].map(ST["station_type"])
r["category"] = pd.Categorical(r["category"], CATS, ordered=True)
# History only: future bookings (status "booked", picked up on or after day 1) are left out of all exploration.
# `open` rentals stay in: they were picked up before day 1 and are part of the observed past demand.
r = r[r["status"] != "booked"].reset_index(drop=True)
print(r["status"].value_counts().to_string())
print("pickup dates:", r["pickup_date"].min().date(), "...", r["pickup_date"].max().date())


def grid_axes(nrows, ncols, w=3.6, h=2.0, **kw):
    """Small-multiples grid: stations as rows, categories as columns."""
    fig, axes = plt.subplots(nrows, ncols, figsize=(w * ncols, h * nrows), squeeze=False, **kw)
    return fig, axes


def label_grid(axes, rows=STATIONS, cols=CATS):
    for j, k in enumerate(cols):
        axes[0, j].set_title(k, fontsize=11, fontweight="bold", color=CAT_COLOR.get(k, "k"))
    for i, s in enumerate(rows):
        axes[i, 0].set_ylabel(ST_LABEL.get(s, s), fontsize=8, rotation=0, ha="right", va="center", labelpad=8)

In [ ]:
hist = r[r["status"] == "completed"].copy()
print(f"{len(hist):,} completed rentals, {hist['length'].sum():,} car-days")
hist.groupby("category", observed=True)["rev_per_day"].describe().round(1)

## 1. Histogram of revenue per car-day, per station × category

One histogram per cell, one count per rental. Columns share the x-axis (one category, all stations).
The dark line is the car-day-weighted mean $\hat b_{s,k}$ (our base estimate); the box gives $n$, $\hat b$ and the
coefficient of variation.
Set `WEIGHT_BY_CAR_DAYS = True` to weight each rental by its length (the histogram that matches $\hat b$).

In [ ]:
WEIGHT_BY_CAR_DAYS = False

base = (hist.groupby(["pickup_station_id", "category"], observed=True)
            .apply(lambda g: g["revenue"].sum() / g["length"].sum(), include_groups=False)
            .rename("base"))

fig, axes = grid_axes(len(STATIONS), len(CATS), w=3.8, h=1.7)
for j, k in enumerate(CATS):
    col = hist[hist["category"] == k]["rev_per_day"]
    bins = np.linspace(col.min(), col.max(), 40)
    for i, s in enumerate(STATIONS):
        ax = axes[i, j]
        g = hist[(hist["pickup_station_id"] == s) & (hist["category"] == k)]
        w = g["length"] if WEIGHT_BY_CAR_DAYS else None
        ax.hist(g["rev_per_day"], bins=bins, weights=w, color=CAT_COLOR[k], edgecolor="white", linewidth=0.5)
        b = base.loc[(s, k)]
        ax.axvline(b, color="#0b0b0b", lw=1.2)
        cv = g["rev_per_day"].std() / g["rev_per_day"].mean()
        ax.text(0.98, 0.95, f"n={len(g)}\nb̂={b:.0f}\ncv={cv:.2f}", transform=ax.transAxes,
                ha="right", va="top", fontsize=7, color="#52514e")
        ax.set_xlim(bins[0], bins[-1])
        if i < len(STATIONS) - 1:
            ax.tick_params(labelbottom=False)
label_grid(axes)
for ax in axes[-1]:
    ax.set_xlabel("revenue per car-day")
fig.suptitle("Revenue per car-day per rental (completed), by pickup station × category", y=1.0, fontsize=12)
fig.tight_layout()
plt.show()

## 2. Base estimate table

$\hat b_{s,k}$, the number of rentals, the per-rental CV, and the standard error of $\hat b$
(ratio estimator, $\mathrm{SE} \approx \sqrt{\sum_i L_i^2 (r_i-\hat b)^2}\,/\sum_i L_i$ with $r_i = R_i/L_i$).

In [ ]:
def cell_stats(g):
    b = g["revenue"].sum() / g["length"].sum()
    se = np.sqrt((g["length"] ** 2 * (g["rev_per_day"] - b) ** 2).sum()) / g["length"].sum()
    return pd.Series({"base": b, "n": len(g), "car_days": g["length"].sum(),
                      "cv": g["rev_per_day"].std() / g["rev_per_day"].mean(), "se": se, "rel_se": se / b})

stats = hist.groupby(["pickup_station_id", "category"], observed=True).apply(cell_stats, include_groups=False)
print("Base revenue per car-day b̂[s,k]:")
display(stats["base"].unstack().round(1))
print("Relative standard error of b̂ (in %):")
display((100 * stats["rel_se"]).unstack().round(2))
print("Per-rental CV of revenue per car-day:")
display(stats["cv"].unstack().round(3))

The pipeline uses `fleetopt.forecast.base_revenue(inst)` (also written to `estimates/base_revenue.csv` by
`scripts/estimate_base_revenue.py`). Check that it gives the same numbers as this notebook:

In [ ]:
from fleetopt import forecast

fb = forecast.base_revenue(inst)
nb = (stats[["base", "se"]].reset_index().rename(columns={"pickup_station_id": "station_id"})
        .astype({"category": str}))
cmp = fb.merge(nb, on=["station_id", "category"], suffixes=("", "_nb"))
diff = max((cmp["base_cell"] - cmp["base_nb"]).abs().max(), (cmp["se"] - cmp["se_nb"]).abs().max())
assert len(cmp) == 40 and diff < 1e-9, diff
print(f"fleetopt.forecast.base_revenue matches this notebook (40 cells, max |diff| = {diff:.1e})")

## 3. Is base multiplicative, $b_{s,k} \approx \alpha_s \beta_k$?

Two-way decomposition of $\log \hat b_{s,k}$ into station and category effects. The heat map shows the cell
relative to the fit, $\hat b_{s,k} / (\alpha_s\beta_k)$. If it is within a few SE of 1 everywhere, a multiplicative model
(10 + 4 parameters instead of 40) is a fine shrinkage target, and the cells with few rentals can borrow from it.

In [ ]:
lb = np.log(stats["base"].unstack())
mu = lb.values.mean()
a = lb.mean(axis=1) - mu          # station effect
c = lb.mean(axis=0) - mu          # category effect
fit = np.exp(mu + a.values[:, None] + c.values[None, :])
ratio = stats["base"].unstack() / fit
z = (ratio - 1) / stats["rel_se"].unstack()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
im = axes[0].imshow(ratio.values, cmap="RdBu_r", vmin=0.95, vmax=1.05, aspect="auto")
axes[0].set_title("b̂ / (α_s β_k)"); fig.colorbar(im, ax=axes[0], shrink=0.8)
im = axes[1].imshow(z.values, cmap="RdBu_r", vmin=-4, vmax=4, aspect="auto")
axes[1].set_title("(ratio − 1) / rel. SE"); fig.colorbar(im, ax=axes[1], shrink=0.8)
for ax, M, fmt in [(axes[0], ratio.values, "{:.3f}"), (axes[1], z.values, "{:+.1f}")]:
    ax.set_xticks(range(len(CATS)), CATS, rotation=30); ax.set_yticks(range(len(STATIONS)), STATIONS); ax.grid(False)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, fmt.format(M[i, j]), ha="center", va="center", fontsize=7)
fs = np.exp(a)
axes[2].barh(STATIONS, fs.values, color=[TYPE_COLOR[ST.loc[s, "station_type"]] for s in STATIONS])
axes[2].axvline(1, color="#52514e", lw=0.8); axes[2].invert_yaxis()
axes[2].set_title("station factor α_s (geo-mean = 1)\nblue = airport, orange = downtown")
fig.tight_layout(); plt.show()
print("category factor β_k × exp(μ):", (np.exp(mu + c)).round(1).to_dict())
print(f"max |ratio − 1| = {np.abs(ratio.values - 1).max():.3%},  max |z| = {np.abs(z.values).max():.1f}")

## 4. Shape of the per-rental noise

Residual $\varepsilon_i = r_i / \hat b_{s,k}$, pooled over cells. Normal vs. log-normal matters only for simulation
(our own scorer), since revenue is linear in base. Also: is the noise the same in every category / station type?

In [ ]:
hist["eps"] = hist["rev_per_day"] / hist.set_index(["pickup_station_id", "category"]).index.map(base).to_numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
ax = axes[0]
ax.hist(hist["eps"], bins=80, density=True, color="#2a78d6", edgecolor="white", linewidth=0.3)
m, sd = hist["eps"].mean(), hist["eps"].std()
x = np.linspace(hist["eps"].min(), hist["eps"].max(), 300)
ax.plot(x, np.exp(-0.5 * ((x - m) / sd) ** 2) / (sd * np.sqrt(2 * np.pi)), color="#0b0b0b", lw=1.5, label="normal fit")
ls = np.log(hist["eps"]).std(); lm = np.log(hist["eps"]).mean()
ax.plot(x, np.exp(-0.5 * ((np.log(x) - lm) / ls) ** 2) / (x * ls * np.sqrt(2 * np.pi)), color="#eb6834", lw=1.5, ls="--", label="log-normal fit")
ax.set_title(f"r_i / b̂ pooled: mean {m:.3f}, sd {sd:.3f}"); ax.legend(fontsize=7)

# QQ plots against normal and log-normal
qs = np.linspace(0.001, 0.999, 400)
from statistics import NormalDist
zq = np.array([NormalDist().inv_cdf(q) for q in qs])
axes[1].plot(zq, np.quantile((hist["eps"] - m) / sd, qs), ".", ms=2, color="#2a78d6", label="ε (normal)")
le = np.log(hist["eps"])
axes[1].plot(zq, np.quantile((le - le.mean()) / le.std(), qs), ".", ms=2, color="#eb6834", label="log ε (log-normal)")
axes[1].plot([-3.2, 3.2], [-3.2, 3.2], color="#52514e", lw=0.8)
axes[1].set_title("QQ vs. standard normal"); axes[1].legend(fontsize=7); axes[1].set_xlabel("normal quantile")

sd_tab = hist.groupby(["station_type", "category"], observed=True)["eps"].std().unstack()
xx = np.arange(len(CATS)); wbar = 0.38
for t_i, t in enumerate(["airport", "downtown"]):
    axes[2].bar(xx + (t_i - 0.5) * wbar, sd_tab.loc[t].values, width=wbar - 0.02, color=TYPE_COLOR[t], label=t)
axes[2].set_xticks(xx, CATS); axes[2].set_title("sd of r_i / b̂ by station type × category"); axes[2].legend(fontsize=7)
fig.tight_layout(); plt.show()
display(sd_tab.round(3))

## 5. Does revenue per car-day depend on anything else?

Mean of $r_i/\hat b_{s,k}$ (car-day weighted) by rental length, pickup weekday, lead time, history week and status,
with 95 % intervals. Flat at 1 means: base depends only on $(s,k)$. Open rentals are included here
(normalized by the same $\hat b$ from completed rentals) to see whether they are priced alike.

In [ ]:
allr = r.copy()
allr["eps"] = allr["rev_per_day"] / allr.set_index(["pickup_station_id", "category"]).index.map(base).to_numpy()
allr["lead_band"] = pd.cut(allr["lead"], [-1, 0, 1, 3, 7, 14, 100], labels=["0", "1", "2-3", "4-7", "8-14", "15+"])
H = allr[allr["status"] == "completed"]

def wmean_ci(g):
    w = g["length"]; m = (w * g["eps"]).sum() / w.sum()
    se = np.sqrt((w ** 2 * (g["eps"] - m) ** 2).sum()) / w.sum()
    return pd.Series({"m": m, "lo": m - 1.96 * se, "hi": m + 1.96 * se, "n": len(g)})

panels = [("length", H, "rental length (car-days)"), ("weekday", H, "pickup weekday"),
          ("lead_band", H, "lead time (days)"), ("week", H, "history week"), ("status", allr, "status")]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.2), sharey=True)
for ax, (col, data, title) in zip(axes, panels):
    t = data.assign(_grp=data[col]).groupby("_grp", observed=True).apply(wmean_ci, include_groups=False)  # copy: groupby drops its key column
    xs = np.arange(len(t))
    ax.errorbar(xs, t["m"], yerr=[t["m"] - t["lo"], t["hi"] - t["m"]], fmt="o", ms=5, color="#2a78d6", ecolor="#9a9993", capsize=2)
    ax.axhline(1, color="#52514e", lw=0.8)
    ax.set_xticks(xs, [str(v) for v in t.index], rotation=0 if len(t) < 9 else 90)
    ax.set_title(title)
axes[0].set_ylabel("r_i / b̂  (car-day weighted)")
fig.tight_layout(); plt.show()

## 6. Does base move over time, and together with demand?

The hidden scenarios draw base "of that scenario" (R8). If base had a common shock per week or per day, the
week means of $r_i/\hat b$ would vary more than per-rental noise alone explains. Test: $z = (\bar\varepsilon_{\text{grp}} - 1)/\mathrm{SE}$
per (station, category, week) and per (station, category, pickup date). $\mathrm{Var}(z) \approx 1$ means no extra shock.

Then the [[Project Plan]] to-do: **corr(base, demand)**. Per (station, category, week) and per (station, category, date),
compare the price level $\bar\varepsilon$ with the number of pickups relative to the cell's weekday mean.

In [ ]:
SD_EPS = H["eps"].std()   # pooled per-rental sd (section 4 shows it is the same in every cell)

def group_z(keys):
    """Car-day-weighted mean of eps per group, and z against pure per-rental noise (pooled sd, not the
    group's own sample sd, which is too noisy for groups of a few rentals)."""
    w = H["length"]
    g = (H.assign(we=w * H["eps"], w=w, w2=w ** 2)
           .groupby(keys, observed=True)[["we", "w", "w2"]].sum())
    g["n"] = H.groupby(keys, observed=True).size()
    g["m"] = g["we"] / g["w"]
    se = SD_EPS * np.sqrt(g["w2"]) / g["w"]
    return g.assign(z=(g["m"] - 1) / se)

zw = group_z(["pickup_station_id", "category", "week"])
zd = group_z(["pickup_station_id", "category", "pickup_date"])
print(f"per (s,k,week): groups={len(zw)}, Var(z) = {zw['z'].var():.2f}, sd of week means = {zw['m'].std():.4f}")
print(f"per (s,k,date): groups={len(zd)}, Var(z) = {zd['z'].var():.2f}, sd of day means  = {zd['m'].std():.4f}")

# Pickups relative to the (s, k, weekday) mean, vs. price level of the same group
pk = H.groupby(["pickup_station_id", "category", "pickup_date"], observed=True).size().rename("pickups").reset_index()
pk["weekday"] = pk["pickup_date"].dt.day_name().str[:3]
pk["week"] = (pk["pickup_date"] - H["pickup_date"].min()).dt.days // 7 + 1
pk["pick_rel"] = pk["pickups"] / pk.groupby(["pickup_station_id", "category", "weekday"], observed=True)["pickups"].transform("mean")
dd = zd.reset_index().merge(pk, on=["pickup_station_id", "category", "pickup_date"])
wk = pk.groupby(["pickup_station_id", "category", "week"], observed=True)["pickups"].sum().rename("pickups").reset_index()
wk["pick_rel"] = wk["pickups"] / wk.groupby(["pickup_station_id", "category"], observed=True)["pickups"].transform("mean")
ww = zw.reset_index().merge(wk, on=["pickup_station_id", "category", "week"])

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].hist(zd["z"], bins=50, density=True, color="#2a78d6", edgecolor="white", linewidth=0.3, label="per (s,k,date)")
xg = np.linspace(-4, 4, 200); axes[0].plot(xg, np.exp(-xg ** 2 / 2) / np.sqrt(2 * np.pi), color="#0b0b0b", lw=1.2, label="N(0,1)")
axes[0].set_title("z of daily price level"); axes[0].legend(fontsize=7)
for ax, d, lab in [(axes[1], ww, "week"), (axes[2], dd, "pickup date")]:
    for k in CATS:
        e = d[d["category"] == k]
        ax.scatter(e["pick_rel"], e["m"], s=8, alpha=0.6, color=CAT_COLOR[k], label=k)
    rho = np.corrcoef(d["pick_rel"], d["m"])[0, 1]
    ax.set_title(f"price level vs. pickups, per (s,k,{lab}): ρ = {rho:+.3f}")
    ax.set_xlabel("pickups / cell mean"); ax.set_ylabel("mean r_i / b̂")
axes[1].legend(fontsize=7, markerscale=2)
fig.tight_layout(); plt.show()

## 7. Takeaways (first run, 2026-11-13)

- **Base is $\hat b_{s,k}$ = Σ revenue / Σ car-days, and it is precise.** Relative SE 0.5–1.8 % per cell, which is
  negligible next to the demand uncertainty.
- **Exactly multiplicative:** $b_{s,k} = \alpha_s\beta_k$ to within 1.6 %, every cell within 1.6 SE of the fit.
  Station factors run from 0.67 (S05) to 1.24 (S09); airports are average (S01 1.03, S06 0.99).
  The 14-parameter fit is a free shrinkage target, but with these SEs the raw cell estimate is just as good.
- **Per-rental noise:** CV ≈ 0.15 in every station, category and station type; right-skewed, closer to log-normal
  than normal. This only matters for simulating revenue, not for the expected-profit model.
- **No other effects:** flat in rental length, pickup weekday, lead time and history week (week 8 is −0.9 %, ≈2.5 SE,
  the only borderline point). Open rentals are priced like the completed ones.
- **No time shocks, no link to demand:** week and day price levels vary exactly as per-rental noise predicts
  (Var(z) ≈ 0.85 weekly, 0.97 daily), and their correlation with pickups is ≈ 0 (+0.04 weekly, +0.01 daily).
  So $E[b\,D] = E[b]\,E[D]$, and plugging in $\hat b_{s,k}$ deterministically is right for the expected-profit objective.
- Caveat: the history can't show how much the hidden scenarios vary base *between* scenarios. That doesn't change
  the expected profit (linear in $b$); it only widens the spread of the 30-scenario score.